# Fall 2024 Data Science Track: Week 2 - Data Cleaning Exercise

## Packages, Packages, Packages!

Import *all* the things here! You need the standard stuff: `pandas` and `numpy`.

If you got more stuff you want to use, add them here too. 🙂

In [1]:
# Install pandas and numpy if the active Python environment does not have them. Using Python venv is recommended when you are doing so.
%pip install pandas numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import here.
import pandas as pd
import numpy as np


## Introduction

With the packages out of the way, now you will be working with the following data sets:

* `food_coded.csv`: [Food choices](https://www.kaggle.com/datasets/borapajo/food-choices?select=food_coded.csv) from Kaggle
* `Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv`: [Ask A Manager Salary Survey 2021 (Responses)](https://docs.google.com/spreadsheets/d/1IPS5dBSGtwYVbjsfbaMCYIWnOuRmJcbequohNxCyGVw/view?&gid=1625408792) as *Tab Separated Values (.tsv)* from Google Docs

Each one poses different challenges. But you’ll―of course―overcome them with what you learned in class! 😉

## Food Choices Data Set

### Load the Data

Load the Food choices data set into a new variable, `df_food`.

In [3]:
# Load the Food choices data set.

food_data_set_path = "../data/food_coded.csv"

df_food = pd.read_csv(food_data_set_path)


### Explore the Data

How much data did you just load?

In [4]:
# Count by hand. (lol kidding)
print("rows:", len(df_food))
print("columns:", df_food.shape[1])
df_food.shape


rows: 125
columns: 61


(125, 61)

In [5]:
# Try to print or display it nicely.
df_food.head()


,GPA,Gender,breakfast,calories_chicken,calories_day,calories_scone,coffee,comfort_food,comfort_food_reasons,comfort_food_reasons_coded,...,soup,sports,thai_food,tortilla_calories,turkey_calories,type_sports,veggies_day,vitamins,waffle_calories,weight
0,2.4,2,1,430,NaN,315.0,1,none,we dont have comfort,9.0,...,1.0,1.0,1,1165.0,345,car racing,5,1,1315,187
1,3.654,1,1,610,3.0,420.0,2,"chocolate, chips, ice cream","Stress, bored, anger",1.0,...,1.0,1.0,2,725.0,690,Basketball,4,2,900,155
2,3.3,1,1,720,4.0,420.0,2,"frozen yogurt, pizza, fast food","stress, sadness",1.0,...,1.0,2.0,5,1165.0,500,none,5,1,900,I'm not answering this.
3,3.2,1,1,430,3.0,420.0,2,"Pizza, Mac and cheese, ice cream",Boredom,2.0,...,1.0,2.0,5,725.0,690,NaN,3,1,1315,"Not sure, 240"
4,3.5,1,1,720,2.0,420.0,2,"Ice cream, chocolate, chips","Stress, boredom, cravings",1.0,...,1.0,1.0,4,940.0,500,Softball,4,2,760,190


What are the columns and their types in this data set?

In [6]:
# Show the column names and their types.
df_food.dtypes


GPA                     str
Gender                int64
breakfast             int64
calories_chicken      int64
calories_day        float64
                     ...   
type_sports             str
veggies_day           int64
vitamins              int64
waffle_calories       int64
weight                  str
Length: 61, dtype: object

In [7]:
pd.set_option("display.max_rows", 100) # Set pandas to render up to 100 rows first. Default caps to 10 and takes the middle chunk out to make the DataFrame fit.

# Try to display the column information as a nice pandas DataFrame.
df_food.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"})


,column,dtype
0,GPA,str
1,Gender,int64
2,breakfast,int64
3,calories_chicken,int64
4,calories_day,float64
5,calories_scone,float64
6,coffee,int64
7,comfort_food,str
8,comfort_food_reasons,str
9,comfort_food_reasons_coded,float64


### Clean the Data

Perhaps we’d like to know more another day, but the team is really interested in just the relationship between calories (`calories_day`) and weight. …and maybe gender.

Can you remove the other columns? (Assign the result to a new variable, `df_food_col_subset`.)

In [8]:
# Remove ‘em.
keep_cols = ["calories_day", "weight", "Gender"]
df_food_col_subset = df_food[keep_cols].copy()
df_food_col_subset


,calories_day,weight,Gender
0,NaN,187,2
1,3.0,155,1
2,4.0,I'm not answering this.,1
3,3.0,"Not sure, 240",1
4,2.0,190,1
...,...,...,...
120,4.0,156,1
121,2.0,180,1
122,NaN,120,1
123,4.0,135,2


In [9]:
# Is there a second way to remove columns?
df_food_col_subset = df_food.drop(
    columns=[c for c in df_food.columns if c not in keep_cols]
).copy()
df_food_col_subset.head()


,Gender,calories_day,weight
0,2,NaN,187
1,1,3.0,155
2,1,4.0,I'm not answering this.
3,1,3.0,"Not sure, 240"
4,1,2.0,190


In [10]:
# 🚀 Extra credit: What about a third way to remove columns?
df_food_col_subset = df_food.loc[:, keep_cols].copy()
df_food_col_subset.head()


,calories_day,weight,Gender
0,NaN,187,2
1,3.0,155,1
2,4.0,I'm not answering this.,1
3,3.0,"Not sure, 240",1
4,2.0,190,1


What about `NaN`s? How many are there?

In [11]:
# Count ‘em.
df_food_col_subset.isna().sum()


calories_day    19
weight           2
Gender           0
dtype: int64

In [12]:
# 🚀 Extra credit: Try to display the NaN sums as a nice pandas DataFrame.
df_food_col_subset.isna().sum().to_frame(name="nan_count")


,nan_count
calories_day,19
weight,2
Gender,0


In [13]:
# 🚀 Extra credit: You can turn one line of code into multiple lines of code using a backslash as a continuation character to break up what would otherwise be a long line.
nan_counts = df_food_col_subset.isna().sum().to_frame(name="nan_count") \
    .sort_values("nan_count", ascending=False)
nan_counts


,nan_count
calories_day,19
weight,2
Gender,0


We gotta remove those `NaN`s―the entire row.

In [14]:
# Drop ‘em in-place.
df_food_col_subset.dropna(inplace=True)
df_food_col_subset.shape


(104, 3)

In [15]:
# 🚀 Extra credit: Check if the in-place modifications also affected the original DataFrame, df_food. 🙂
print("Subset rows after dropna:", len(df_food_col_subset))
print("Original df_food rows:", len(df_food))
print("Original calories_day NaNs:", df_food["calories_day"].isna().sum())
print("Because we used .copy(), dropping rows in the subset did not change df_food.")


Subset rows after dropna: 104
Original df_food rows: 125
Original calories_day NaNs: 19
Because we used .copy(), dropping rows in the subset did not change df_food.


But what about the weird non-numeric values in the column obviously meant for numeric data?

Notice the data type of that column from when you got the types of all the columns?

If only we could convert the column to a numeric type and drop the rows with invalid values. 🤔

In [16]:
# Fix that in-place.
df_food_col_subset["weight"] = pd.to_numeric(
    df_food_col_subset["weight"], errors="coerce"
)
df_food_col_subset.dropna(inplace=True)
df_food_col_subset.dtypes


calories_day    float64
weight          float64
Gender            int64
dtype: object

Now this data seems reasonably clean for our purposes! 😁

Let’s save it somewhere to be shipped off to another teammate. 💾

In [17]:
# Savey save!
food_cleaned_path = "../data/food_calories_weight_gender_clean.csv"
df_food_col_subset.to_csv(food_cleaned_path, index=False)


In [18]:
# 🚀 Extra credit: Load and print a few lines of the saved file as plain text.
with open(food_cleaned_path, encoding="utf-8") as f:
    print("".join(f.readlines()[:8]))


calories_day,weight,Gender
3.0,155.0,1
2.0,190.0,1
3.0,190.0,1
3.0,180.0,2
3.0,137.0,1
3.0,125.0,1
3.0,116.0,1



## Ask a Manager Salary Survey 2021 (Responses) Data Set

### Load the Data

Load the Ask A Manager Salary Survey 2021 (Responses) data set into a new variable, `df_salary`.

In [19]:
# Load the Ask A Manager Salary Survey 2021 (Responses) data set.
salary_data_set_path = "../data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv"
df_salary = pd.read_csv(salary_data_set_path, sep="\t")
df_salary.head()


,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White
3,4/27/2021 11:02:41,25-34,Nonprofits,Program Manager,NaN,"62,000",3000.0,USD,NaN,NaN,USA,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White
4,4/27/2021 11:02:42,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,"60,000",7000.0,USD,NaN,NaN,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White


Was that hard? 🙃

### Explore

You know the drill.

How much data did you just load?

In [20]:
# Count by hand. I’m dead serious.
print("rows:", len(df_salary))
print("columns:", df_salary.shape[1])
df_salary.shape


rows: 28062
columns: 18


(28062, 18)

What are the columns and their types?

In [21]:
# Show the column names and their types.
df_salary.dtypes


Timestamp                                                                                                                                                                                                                                   str
How old are you?                                                                                                                                                                                                                            str
What industry do you work in?                                                                                                                                                                                                               str
Job title                                                                                                                                                                                                                                   str
If your job title needs additional conte

Oh… Ugh! Give these columns easier names to work with first. 🙄

In [22]:
# Rename ‘em in-place.
# Non-binding suggestions: timestamp, age, industry, title, title_context, salary, additional_compensation, currency, other_currency, salary_context, country, state, city, total_yoe, field_yoe, highest_education_completed	gender, race
df_salary.columns = [
    "timestamp",
    "age",
    "industry",
    "title",
    "title_context",
    "salary",
    "additional_compensation",
    "currency",
    "other_currency",
    "salary_context",
    "country",
    "state",
    "city",
    "total_yoe",
    "field_yoe",
    "highest_education_completed",
    "gender",
    "race",
]
df_salary.head()


,timestamp,age,industry,title,title_context,salary,additional_compensation,currency,other_currency,salary_context,country,state,city,total_yoe,field_yoe,highest_education_completed,gender,race
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White
3,4/27/2021 11:02:41,25-34,Nonprofits,Program Manager,NaN,"62,000",3000.0,USD,NaN,NaN,USA,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White
4,4/27/2021 11:02:42,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,"60,000",7000.0,USD,NaN,NaN,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White


It’s a lot, and that should not have been easy. 😏

You’re going to have a gander at the computing/tech subset first because thats *your* industry. But first, what value corresponds to that `industry`?

In [23]:
# List the unique industries and a count of their instances.
df_salary["industry"].value_counts()


industry
Computing or Tech                               4699
Education (Higher Education)                    2464
Nonprofits                                      2419
Health care                                     1896
Government and Public Administration            1889
                                                ... 
Undergrad student                                  1
Concrete Construction                              1
I'm currently a student and don't have a job       1
Student                                            1
Wine & Spirits                                     1
Name: count, Length: 1219, dtype: int64

That value among the top 5 is what you’re looking for innit? Filter out all the rows not in that industry and save it into a new variable, `df_salary_tech`. 

In [24]:
# Filtery filter.
df_salary_tech = df_salary[df_salary["industry"] == "Computing or Tech"].copy()
df_salary_tech.head()


,timestamp,age,industry,title,title_context,salary,additional_compensation,currency,other_currency,salary_context,country,state,city,total_yoe,field_yoe,highest_education_completed,gender,race
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
8,4/27/2021 11:03:01,45-54,Computing or Tech,Systems Analyst,Data developer/ETL Developer,"112,000",10000.0,USD,NaN,NaN,US,Missouri,St. Louis,21 - 30 years,21 - 30 years,College degree,Woman,White
43,4/27/2021 11:04:04,25-34,Computing or Tech,Principal Software Engineer,NaN,"187,500",5000.0,USD,NaN,NaN,United States,Pennsylvania,Pittsburgh,8 - 10 years,5-7 years,College degree,Woman,White
44,4/27/2021 11:04:04,25-34,Computing or Tech,Intelligence Analyst,NaN,"110,000",20000.0,USD,NaN,"Around 20,000 a year in stock",USA,Virginia,"Arlington, VA",8 - 10 years,8 - 10 years,Master's degree,Man,White
46,4/27/2021 11:04:07,35-44,Computing or Tech,Mobile developer,NaN,"144,600",2500.0,USD,NaN,NaN,USA,Massachusetts,Boston,5-7 years,5-7 years,PhD,Woman,White


Do a sanity check by counting.

In [25]:
# Sanity check count.
len(df_salary_tech)


4699

We are very interested in salary figures. But how many dollars 💵 is a euro 💶 or a pound 💷? That sounds like a problem for another day. 🫠

For now, let’s just look at U.S. dollars (`'USD'`).

In [26]:
# Filtery filter (in place) for just the jobs that pay in USD!
df_salary_tech.query("currency == 'USD'", inplace=True)
df_salary_tech["currency"].value_counts()


currency
USD    3777
Name: count, dtype: int64

What we really want know is how each U.S. state pays in tech. What value in `country` represents the United States of America?

In [27]:
# We did filter for USD, so if we do a count of each unique country in descending count order, the relevant value(s) should show up at the top.
df_salary_tech["country"].value_counts()


country
United States                1576
USA                          1222
US                            412
U.S.                          108
United States of America       90
United States                  68
Usa                            59
USA                            56
usa                            28
United states                  23
united states                  14
Us                             12
us                              9
U.S.A.                          7
United States of America        7
Israel                          5
Canada                          4
U.S.                            2
United State of America         2
Unite States                    2
Australia                       2
UnitedStates                    2
India                           2
U.S                             2
Usa                             2
United States Of America        2
Spain                           2
Brazil                          2
United Kingdom                  2
united

### Clean the Data

Well, we can’t get our answers with what we currently have, so you’ll have to make some changes.

Let’s not worry about anything below the first 5 values for now. Convert the top 5 to a single canonical value―say, `'US'`, which is nice and short.

In [28]:
# Replace them all in-place with 'US'.
top_countries = df_salary_tech["country"].value_counts().head(5).index
df_salary_tech["country"] = df_salary_tech["country"].replace(top_countries, "US")


Have a look at the count of each unique country again now.

In [29]:
# Count again.
df_salary_tech["country"].value_counts()


country
US                           3408
United States                  68
Usa                            59
USA                            56
usa                            28
United states                  23
united states                  14
Us                             12
us                              9
U.S.A.                          7
United States of America        7
Israel                          5
Canada                          4
U.S.                            2
United State of America         2
Unite States                    2
Australia                       2
UnitedStates                    2
India                           2
U.S                             2
Usa                             2
United States Of America        2
Spain                           2
Brazil                          2
United Kingdom                  2
united States                   2
New Zealand                     2
Poland                          2
France                          2
U.S.A 

Did you notice anything interesting?

In [30]:
# 🚀 Extra credit: Resolve [most of] those anomalous cases too without exhaustively taking every variant literally into account.
country_norm = (
    df_salary_tech["country"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)
us_like = country_norm.str.contains(
    r"united states|united state|\busa\b|^us$|^u s$|^u s a$",
    regex=True,
)
df_salary_tech.loc[us_like, "country"] = "US"


In [31]:

# 🚀 Extra credit: If you’ve resolved it, let’s see how well you did by counting the number of instances of each unique value.
df_salary_tech["country"].value_counts().head(20)


country
US                3715
Israel               5
Canada               4
Unite States         2
Australia            2
UnitedStates         2
India                2
Spain                2
Brazil               2
United Kingdom       2
New Zealand          2
Poland               2
France               2
Uniyed states        1
America              1
Puerto Rico          1
Cuba                 1
Danmark              1
Italy                1
International        1
Name: count, dtype: int64

It’s looking good so far. Let’s find out the minimum, mean, and maximum (in that order) salary by state, sorted by the mean in descending order.

In [32]:
# Find the minimum, mean, and maximum salary in USD by U.S. state.
df_salary_us_tech = df_salary_tech[df_salary_tech["country"] == "US"].copy()
df_salary_us_tech.groupby("state")["salary"].agg(["min", "mean", "max"]).sort_values(
    "mean", ascending=False
)


TypeError: dtype 'str' does not support operation 'mean'

 Well, pooh! We forgot that `salary` isn’t numeric. Something wrong must be fixed. 🤔

In [33]:
# Fix it in-place.
df_salary_us_tech["salary"] = pd.to_numeric(
    df_salary_us_tech["salary"].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
)
df_salary_us_tech["salary"].head()


8     112000
43    187500
44    110000
46    144600
47    200850
Name: salary, dtype: int64

Let’s try that again.

In [34]:
# Try it again. Yeah!
df_salary_us_tech.groupby("state")["salary"].agg(["min", "mean", "max"]).sort_values(
    "mean", ascending=False
)


,min,mean,max
state,,,
"Michigan, Texas, Washington",340000,340000.000000,340000
"California, Oregon",200000,200000.000000,200000
"California, Colorado",176000,176000.000000,176000
"Georgia, Massachusetts",175000,175000.000000,175000
Florida,28800,157457.232143,2600000
"Alabama, District of Columbia",156000,156000.000000,156000
California,0,155224.893130,875000
Washington,72,151486.867257,950000
New York,14000,148157.669540,590000


That did the trick! Now let’s narrow this to data 2021 and 2022 just because (lel). *(Hint: that timestamp column may not be a temporal type right now.)*

In [35]:
# Filter the data to within 2021, 2022, or 2023, saving the DataFrame to a new variable, and generate the summary again.
df_salary_us_tech["timestamp"] = pd.to_datetime(df_salary_us_tech["timestamp"])
df_salary_us_tech_recent = df_salary_us_tech[
    df_salary_us_tech["timestamp"].dt.year.isin([2021, 2022, 2023])
]
df_salary_us_tech_recent.groupby("state")["salary"].agg(["min", "mean", "max"]).sort_values(
    "mean", ascending=False
)


,min,mean,max
state,,,
"Michigan, Texas, Washington",340000,340000.000000,340000
"California, Oregon",200000,200000.000000,200000
"California, Colorado",176000,176000.000000,176000
"Georgia, Massachusetts",175000,175000.000000,175000
"Alabama, District of Columbia",156000,156000.000000,156000
California,0,155355.206422,875000
Washington,72,151635.792899,950000
New York,14000,148157.669540,590000
Nevada,38000,141310.000000,425000


## Bonus

Clearly, we do not have enough data to produce useful figures for the level of specificity you’ve now reached. What do you notice about Delaware and West Virginia?

Let’s back out a bit and return to `df_salary` (which was the loaded data with renamed columns but *sans* filtering).

### Bonus #0

Apply the same steps as before to `df_salary`, but do not filter for any specific industry. Do perform the other data cleaning stuff, and get to a point where you can generate the minimum, mean, and maximum by state.

In [36]:
# Bonus #0: same cleaning on all industries, not just tech.
df_salary_all = df_salary.copy()
df_salary_all.query("currency == 'USD'", inplace=True)

top_countries_all = df_salary_all["country"].value_counts().head(5).index
df_salary_all["country"] = df_salary_all["country"].replace(top_countries_all, "US")

country_norm_all = (
    df_salary_all["country"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)
us_like_all = country_norm_all.str.contains(
    r"united states|united state|\busa\b|^us$|^u s$|^u s a$",
    regex=True,
)
df_salary_all.loc[us_like_all, "country"] = "US"

df_salary_us = df_salary_all[df_salary_all["country"] == "US"].copy()
df_salary_us["salary"] = pd.to_numeric(
    df_salary_us["salary"].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
)
df_salary_us.groupby("state")["salary"].agg(["min", "mean", "max"]).sort_values(
    "mean", ascending=False
)


,min,mean,max
state,,,
"Michigan, Texas, Washington",340000,340000.000000,340000
"Indiana, Ohio",245000,245000.000000,245000
Alaska,27040,232275.078125,10000000
"Colorado, Nevada",190000,190000.000000,190000
"California, Montana",185000,185000.000000,185000
...,...,...,...
"Delaware, Pennsylvania",35000,35000.000000,35000
"District of Columbia, Washington",35000,35000.000000,35000
"Alabama, California",29120,29120.000000,29120


### Bonus #1

This time, format the table output nicely (*$12,345.00*) without modifying the values in the `DataFrame`. That is, `df_salary` should be identical before versus after running your code.

(*Hint: if you run into an error about `jinja2` perhaps you need to `pip install` something.*)

In [37]:
# Bonus #1: format as money for display only. Underlying df_salary is unchanged.
salary_stats = (
    df_salary_us.groupby("state")["salary"]
    .agg(["min", "mean", "max"])
    .sort_values("mean", ascending=False)
)
# .map formats the displayed table; it does not write back into df_salary.
salary_stats.map(lambda x: f"${x:,.2f}")


,min,mean,max
state,,,
"Michigan, Texas, Washington","$340,000.00","$340,000.00","$340,000.00"
"Indiana, Ohio","$245,000.00","$245,000.00","$245,000.00"
Alaska,"$27,040.00","$232,275.08","$10,000,000.00"
"Colorado, Nevada","$190,000.00","$190,000.00","$190,000.00"
"California, Montana","$185,000.00","$185,000.00","$185,000.00"
...,...,...,...
"Delaware, Pennsylvania","$35,000.00","$35,000.00","$35,000.00"
"District of Columbia, Washington","$35,000.00","$35,000.00","$35,000.00"
"Alabama, California","$29,120.00","$29,120.00","$29,120.00"


### Bonus #2

Filter out the non-single-states (e.g., `'California, Colorado'`) in the most elegant way possible (i.e., *not* by blacklisting all the bad values).

In [38]:
# Bonus #2: drop multi-state values like 'California, Colorado' without listing them.
df_salary_us_single_state = df_salary_us[
    df_salary_us["state"].notna() & ~df_salary_us["state"].astype(str).str.contains(",")
].copy()
df_salary_us_single_state.groupby("state")["salary"].agg(["min", "mean", "max"]).sort_values(
    "mean", ascending=False
)


,min,mean,max
state,,,
Alaska,27040,232275.078125,10000000
California,0,114502.729521,875000
Washington,72,107869.712712,1260000
District of Columbia,40,106628.117163,1334782
New York,80,105205.871251,3000000
New Jersey,14850,101132.497475,5000044
Massachusetts,155,98769.116678,1650000
Virginia,57,94718.997433,1300000
Connecticut,0,93572.512712,1900000


### Bonus #3

Show the quantiles instead of just minimum, mean, and maximum―say 0%, 5%, 25%, 50%, 75%, 95%, and 100%. Outliers may be deceiving.

Sort by whatever interests you―like say the *50th* percentile.

And throw in a count by state too. It would be interesting to know how many data points contribute to the figures for each state. (*Hint: your nice formatting from Bonus #1 might not work this time around.* 😜)

In [39]:
# Bonus #3: quantiles plus a count, sorted by the 50th percentile.
quantile_stats = df_salary_us_single_state.groupby("state")["salary"].agg(
    count="count",
    p0=lambda s: s.quantile(0.00),
    p5=lambda s: s.quantile(0.05),
    p25=lambda s: s.quantile(0.25),
    p50=lambda s: s.quantile(0.50),
    p75=lambda s: s.quantile(0.75),
    p95=lambda s: s.quantile(0.95),
    p100=lambda s: s.quantile(1.00),
).sort_values("p50", ascending=False)
quantile_stats


,count,p0,p5,p25,p50,p75,p95,p100
state,,,,,,,,
California,2588,0.0,42000.0,72000.00,100470.0,147000.00,220000.00,875000.0
Washington,1180,72.0,39980.0,65000.00,91000.0,135000.00,201100.00,1260000.0
District of Columbia,973,40.0,48000.0,66500.00,90000.0,125000.00,189400.00,1334782.0
New York,2167,80.0,40916.0,64527.50,90000.0,127000.00,210000.00,3000000.0
Massachusetts,1517,155.0,42000.0,64000.00,86000.0,120000.00,180400.00,1650000.0
Maryland,564,0.0,40150.0,60000.00,82000.0,110000.00,165000.00,353200.0
Connecticut,236,0.0,33787.5,61750.00,81900.0,100000.00,162500.00,1900000.0
Delaware,46,35000.0,41704.0,58869.75,81322.5,105750.00,164247.50,220000.0
Virginia,779,57.0,37360.0,58750.00,81000.0,115000.00,180000.00,1300000.0
